# One-Stage OD — **SSD Enhanced Lab** (Backbone switch + mAP eval + EarlyStopping/LR)
This notebook lets you:
- Switch SSD backbone between **SSD300_VGG16** and **SSDLite320 MobileNetV3 Large** (modern, lightweight, strong).
- Run **inference** on a sample image.
- **Fine-tune** with **Early Stopping** and **LR Scheduler**.
- Compute **mAP (COCO)** using `pycocotools` (with a minimal **VOC→COCO** converter if needed).

> Tested with `torch`/`torchvision` recent versions. Uncomment `pip` cells if running on new environments.

## 0) Setup

In [ ]:
# If needed:
# !pip install -q torch torchvision pillow opencv-python matplotlib pycocotools

import os, json, cv2, math, time
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
import torch, torchvision
from torchvision.transforms import functional as F
from torch.utils.data import DataLoader
print("Torch:", torch.__version__, "| Torchvision:", torchvision.__version__)

# Prepare a demo image
os.makedirs('samples', exist_ok=True)
img_path = 'samples/street.jpg'
if not Path(img_path).exists():
    img = Image.new('RGB', (960, 640), (235, 240, 245))
    d = ImageDraw.Draw(img)
    for (x,y,w,h) in [(120,300,120,80),(420,280,90,70),(700,320,160,100)]:
        d.rectangle([x,y,x+w,y+h], outline=(60,60,60), width=4)
    d.text((20,20), "Dummy Street-like Scene", fill=(10,10,10))
    img.save(img_path)
print("Sample at:", img_path)

## 1) Choose Backbone & Inference

In [ ]:
from torchvision.models.detection import (
    ssd300_vgg16, SSD300_VGG16_Weights,
    ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
)

BACKBONE = 'ssdlite_mnv3'   # 'vgg16' or 'ssdlite_mnv3'

if BACKBONE == 'vgg16':
    weights = SSD300_VGG16_Weights.DEFAULT
    model = ssd300_vgg16(weights=weights).eval()
    preproc = weights.transforms()
    input_size = (300,300)
elif BACKBONE == 'ssdlite_mnv3':
    weights = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
    model = ssdlite320_mobilenet_v3_large(weights=weights).eval()
    preproc = weights.transforms()
    input_size = (320,320)
else:
    raise ValueError("Unknown BACKBONE")

print("Using backbone:", BACKBONE, "| Expected input:", input_size)

img = Image.open('samples/street.jpg').convert('RGB')
x = preproc(img).unsqueeze(0)
with torch.no_grad():
    out = model(x)[0]  # dict: boxes/scores/labels

def draw(img_p, boxes, labels=None, scores=None, th=0.35):
    im = cv2.imread(img_p)
    for i,(x1,y1,x2,y2) in enumerate(boxes):
        if scores is not None and scores[i] < th: 
            continue
        cv2.rectangle(im,(int(x1),int(y1)),(int(x2),int(y2)),(0,255,0),2)
        txt = ""
        if labels is not None: txt += str(int(labels[i]))
        if scores is not None: txt += f" {scores[i]:.2f}"
        if txt:
            cv2.putText(im, txt, (int(x1),max(0,int(y1)-6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    return im

boxes = out['boxes'].cpu().numpy()
scores = out['scores'].cpu().numpy()
labels = out['labels'].cpu().numpy()
os.makedirs('runs_demo', exist_ok=True)
out_path = f"runs_demo/{BACKBONE}_pred.jpg"
cv2.imwrite(out_path, draw('samples/street.jpg', boxes, labels, scores, th=0.35))
print("Saved:", out_path)

## 2) Fine-tune with EarlyStopping + LR Scheduler

In [ ]:
import xml.etree.ElementTree as ET

# Edit your label set (background must be index 0)
CLASSES = ["__background__", "person","car","bus","bike"]

def parse_voc_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    boxes, labels = [], []
    for obj in root.findall('object'):
        name = obj.find('name').text
        bnd = obj.find('bndbox')
        x1,y1,x2,y2 = [float(bnd.find(t).text) for t in ['xmin','ymin','xmax','ymax']]
        boxes.append([x1,y1,x2,y2])
        labels.append(CLASSES.index(name))
    return torch.tensor(boxes, dtype=torch.float32), torch.tensor(labels, dtype=torch.int64)

class VOCDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, ann_dir, transform=None):
        self.imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        self.img_dir, self.ann_dir = img_dir, ann_dir
        self.transform = transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        fn = self.imgs[i]
        img_path = os.path.join(self.img_dir, fn)
        ann_path = os.path.join(self.ann_dir, os.path.splitext(fn)[0] + '.xml')
        img = Image.open(img_path).convert('RGB')
        boxes, labels = parse_voc_xml(ann_path)
        img = self.transform(img) if self.transform else F.to_tensor(img)
        target = {'boxes': boxes, 'labels': labels}
        return img, target

def collate_fn(batch): return tuple(zip(*batch))

def build_model_for_training(num_classes, backbone='ssdlite_mnv3', trainable_backbone_layers=2):
    if backbone == 'ssdlite_mnv3':
        w = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
        m = ssdlite320_mobilenet_v3_large(weights=w, trainable_backbone_layers=trainable_backbone_layers)
        m.head.classification_head.num_classes = num_classes
        m.head.classification_head.cls_logits = torch.nn.Conv2d(
            m.head.classification_head.cls_logits.in_channels,
            m.head.classification_head.num_anchors * num_classes, 3, padding=1)
        preproc = w.transforms()
    elif backbone == 'vgg16':
        w = SSD300_VGG16_Weights.DEFAULT
        m = ssd300_vgg16(weights=w)
        m.head.classification_head.num_classes = num_classes
        m.head.classification_head.cls_logits = torch.nn.Conv2d(
            1024, m.head.classification_head.num_anchors * num_classes, 3, padding=1)
        preproc = w.transforms()
    else:
        raise ValueError("Unknown backbone")
    return m, preproc

train_img_dir = 'VOC/train/JPEGImages'
train_ann_dir = 'VOC/train/Annotations'
val_img_dir   = 'VOC/val/JPEGImages'
val_ann_dir   = 'VOC/val/Annotations'

if not (os.path.exists(train_img_dir) and os.path.exists(val_img_dir)):
    print("VOC folders not found, skip training. Set correct paths above.")
else:
    num_classes = len(CLASSES)
    model_tr, preproc = build_model_for_training(num_classes, backbone=BACKBONE, trainable_backbone_layers=2)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_tr.to(device)

    train_set = VOCDataset(train_img_dir, train_ann_dir, transform=preproc if BACKBONE=='ssdlite_mnv3' else None)
    val_set   = VOCDataset(val_img_dir, val_ann_dir, transform=preproc if BACKBONE=='ssdlite_mnv3' else None)
    train_loader = DataLoader(train_set, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader   = DataLoader(val_set, batch_size=8, shuffle=False, collate_fn=collate_fn, num_workers=2)

    opt = torch.optim.SGD(model_tr.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
    # Choose one scheduler:
    use_cosine = True
    if use_cosine:
        lr_sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
    else:
        lr_sch = torch.optim.lr_scheduler.StepLR(opt, step_size=8, gamma=0.1)

    PATIENCE = 5
    best_val = float('inf')
    patience_cnt = 0

    def evaluate_loss(model, loader):
        model.eval()
        tot = 0.0; n = 0
        with torch.no_grad():
            for imgs, targets in loader:
                imgs = [i.to(device) for i in imgs]
                targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
                loss_dict = model(imgs, targets)
                loss = sum(loss_dict.values()).item()
                tot += loss; n += 1
        return tot / max(1,n)

    EPOCHS = 20
    for ep in range(EPOCHS):
        model_tr.train()
        for imgs, targets in train_loader:
            imgs = [i.to(device) for i in imgs]
            targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
            loss_dict = model_tr(imgs, targets)
            loss = sum(loss_dict.values())
            opt.zero_grad(); loss.backward(); opt.step()
        lr_sch.step()

        val_loss = evaluate_loss(model_tr, val_loader)
        print(f"Epoch {ep+1}/{EPOCHS}: val_loss={val_loss:.4f}")
        if val_loss < best_val - 1e-4:
            best_val = val_loss; patience_cnt = 0
            torch.save(model_tr.state_dict(), f'ssd_{BACKBONE}_best.pth')
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            print("Early stopping triggered."); break

    print("Best val loss:", best_val)

## 3) mAP (COCO) Evaluation with `pycocotools`

In [ ]:
# !pip install -q pycocotools

import xml.etree.ElementTree as ET

def voc_to_coco_json(img_dir, ann_dir, classes, out_json='coco_gt.json'):
    images, annotations = [], []
    categories = [{"id": i, "name": name} for i, name in enumerate(classes) if name != "__background__"]
    ann_id = 1
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    for i,fn in enumerate(img_files,1):
        img_path = os.path.join(img_dir, fn)
        ann_path = os.path.join(ann_dir, os.path.splitext(fn)[0] + '.xml')
        if not os.path.exists(ann_path): 
            continue
        im = Image.open(img_path).convert('RGB')
        w,h = im.size
        images.append({"id": i, "file_name": fn, "width": w, "height": h})
        root = ET.parse(ann_path).getroot()
        for obj in root.findall('object'):
            name = obj.find('name').text
            if name == "__background__": continue
            cid = classes.index(name)  # assumes background=0
            if cid <= 0: continue
            bnd = obj.find('bndbox')
            x1,y1,x2,y2 = [float(bnd.find(t).text) for t in ['xmin','ymin','xmax','ymax']]
            bbox = [x1, y1, x2-x1, y2-y1]
            annotations.append({
                "id": ann_id, "image_id": i, "category_id": cid,
                "bbox": bbox, "area": bbox[2]*bbox[3], "iscrowd": 0
            })
            ann_id += 1
    coco = {"images": images, "annotations": annotations, "categories": categories}
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(coco, f)
    return out_json

def build_model_for_eval(num_classes, backbone='ssdlite_mnv3'):
    if backbone == 'ssdlite_mnv3':
        w = SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
        m = ssdlite320_mobilenet_v3_large(weights=w, trainable_backbone_layers=0)
        m.head.classification_head.num_classes = num_classes
        m.head.classification_head.cls_logits = torch.nn.Conv2d(
            m.head.classification_head.cls_logits.in_channels,
            m.head.classification_head.num_anchors * num_classes, 3, padding=1)
        preproc = w.transforms()
    else:
        w = SSD300_VGG16_Weights.DEFAULT
        m = ssd300_vgg16(weights=w)
        m.head.classification_head.num_classes = num_classes
        m.head.classification_head.cls_logits = torch.nn.Conv2d(
            1024, m.head.classification_head.num_anchors * num_classes, 3, padding=1)
        preproc = w.transforms()
    return m, preproc

def ssd_infer_dataset(model, img_dir, preproc=None, score_th=0.001, device=None):
    model.eval()
    if device is None: device = next(model.parameters()).device
    results = []
    img_files = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
    for idx,fn in enumerate(img_files,1):
        p = os.path.join(img_dir, fn)
        im = Image.open(p).convert('RGB')
        x = preproc(im).unsqueeze(0) if preproc else F.to_tensor(im).unsqueeze(0)
        with torch.no_grad():
            out = model(x.to(device))[0]
        boxes = out['boxes'].cpu().numpy()
        scores = out['scores'].cpu().numpy()
        labels = out['labels'].cpu().numpy()
        for b,s,c in zip(boxes, scores, labels):
            if s < score_th: continue
            results.append({
                "image_id": idx,
                "category_id": int(c),
                "bbox": [float(b[0]), float(b[1]), float(b[2]-b[0]), float(b[3]-b[1])],
                "score": float(s),
            })
    return results

def evaluate_coco_map(gt_json, det_results_json):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    coco_gt = COCO(gt_json)
    coco_dt = coco_gt.loadRes(det_results_json)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    coco_eval.evaluate(); coco_eval.accumulate(); coco_eval.summarize()
    return coco_eval.stats

# Example usage (requires val split to exist)
val_img_dir = 'VOC/val/JPEGImages'
val_ann_dir = 'VOC/val/Annotations'
if os.path.exists(val_img_dir):
    num_classes = len(CLASSES)
    model_eval, preproc_eval = build_model_for_eval(num_classes, backbone=BACKBONE)
    best_path = f'ssd_{BACKBONE}_best.pth'
    if Path(best_path).exists():
        model_eval.load_state_dict(torch.load(best_path, map_location='cpu'))
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_eval.to(device)

    gt_json = voc_to_coco_json(val_img_dir, val_ann_dir, CLASSES, out_json='coco_gt_val.json')
    det = ssd_infer_dataset(model_eval, val_img_dir, preproc=preproc_eval, score_th=0.001, device=device)
    with open('coco_dt_val.json','w',encoding='utf-8') as f: json.dump(det, f)
    print("Wrote coco_gt_val.json and coco_dt_val.json")
    try:
        stats = evaluate_coco_map('coco_gt_val.json', 'coco_dt_val.json')
        print("COCO mAP stats:", stats)
    except Exception as e:
        print("COCO evaluation failed:", e)
else:
    print("Validation folder not found; skip COCO mAP evaluation.")